In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
import os
import warnings

warnings.filterwarnings("ignore")

class SpatialAbsoluteBiasPlotter:
    def __init__(self):
        self.input_dir = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\3-bias_corr\data\rrtm_ncld1"
        self.output_dir = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\3-bias_corr\output_plots\rrtm_ncld1"
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.seasons = ['DJF', 'MAM', 'JJA', 'SON']
        self.extent = [94, 142, -12, 9]

        self.clon = 120.0
        self.clat = -2.5
        proj_std = pyproj.Proj(f"+proj=merc +lon_0={self.clon} +lat_ts={self.clat}")
        _, y_center = proj_std(self.clon, self.clat)
        self.data_crs = ccrs.Mercator(central_longitude=self.clon, latitude_true_scale=self.clat, false_northing=-y_center)

        self.config = {
            'rsds': {'title': 'Surface Downwelling Shortwave Radiation (RSDS)', 'unit': 'W/m2',
                     'abs_vmin': 100, 'abs_vmax': 300, 'abs_cmap': 'YlOrRd', 'bias_vmin': -300, 'bias_vmax': 300},
            't2m': {'title': '2m Temperature (T2M)', 'unit': 'Celcius',
                    'abs_vmin': 22, 'abs_vmax': 29, 'abs_cmap': 'Wistia', 'bias_vmin': -4, 'bias_vmax': 4},
            'ws10': {'title': '10m Wind Speed (WS10)', 'unit': 'm/s',
                     'abs_vmin': 0, 'abs_vmax': 10, 'abs_cmap': 'Blues', 'bias_vmin': -8, 'bias_vmax': 8},
            'ws100': {'title': '100m Wind Speed (WS100)', 'unit': 'm/s',
                      'abs_vmin': 0, 'abs_vmax': 10, 'abs_cmap': 'Blues', 'bias_vmin': -10, 'bias_vmax': 10}
        }

        self.abs_titles = ['ERA5 (Ref)', 'RCM_EC-Earth3', 'Corrected RCM_EC-Earth3', 'RCM_NorESM', 'Corrected RCM_NorESM']
        self.bias_titles = ['RCM_EC-Earth3', 'Corrected RCM_EC-Earth3', 'RCM_NorESM', 'Corrected RCM_NorESM']

    def prep_map(self, ax):
        ax.set_extent(self.extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, edgecolor='black', zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='dimgray', linestyle='--', zorder=2)
        ax.spines['geo'].set_linewidth(0.5)

    def format_layout(self, ax, i, j, season, col_title):
        if j == 0:
            ax.text(-0.06, 0.5, season, va='center', ha='center', rotation=90, 
                    transform=ax.transAxes, fontsize=14, fontweight='bold')
        if i == 0:
            ax.set_title(col_title, fontsize=14, fontweight='bold', pad=8)

    def plot_variable(self, var_key):
        fpath = os.path.join(self.input_dir, f"SPATIAL_SEASONAL_{var_key}_HIST.nc")
        if not os.path.exists(fpath): return

        print(f"🗺️ Rendering Split Matrix for {var_key.upper()}...")
        cfg = self.config[var_key]
        extend_abs = 'max' if 'ws' in var_key else 'both'
        
        with xr.open_dataset(fpath) as ds:
            obs = ds['OBS'].load()
            models = {1: ds['RAW_EC'].load(), 2: ds['COR_EC'].load(), 3: ds['RAW_NOR'].load(), 4: ds['COR_NOR'].load()}

            # ⚡ THE REAL CHEAT V3: KELVIN HACKER UNTUK MATRIKS SPASIAL ⚡
            if var_key == 't2m':
                print("   [INFO] Menjalankan Kelvin Hacker untuk standarisasi Suhu (Celcius)...")
                # 1. Konversi yang masih Kelvin (>200) jadi Celcius
                obs = xr.where(obs > 200, obs - 273.15, obs)
                for k in models.keys():
                    models[k] = xr.where(models[k] > 200, models[k] - 273.15, models[k])
                
                # 2. Obati yang kena Double Subtraction (< -100) kembali ke Celcius
                obs = xr.where(obs < -100, obs + 273.15, obs)
                for k in models.keys():
                    models[k] = xr.where(models[k] < -100, models[k] + 273.15, models[k])

            # 1. RENDER PLOT ABSOLUTE
            fig_abs, axes_abs = plt.subplots(4, 5, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})
            for i, season in enumerate(self.seasons):
                obs_season = obs.sel(season=season)
                for j in range(5):
                    ax = axes_abs[i, j]
                    self.prep_map(ax)
                    data_to_plot = obs_season if j == 0 else models[j].sel(season=season)
                    data_to_plot.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                             vmin=cfg['abs_vmin'], vmax=cfg['abs_vmax'], 
                                             cmap=cfg['abs_cmap'], add_colorbar=False, 
                                             interpolation='bilinear', zorder=1, add_labels=False)
                    self.format_layout(ax, i, j, season, self.abs_titles[j])

            plt.subplots_adjust(wspace=0.01, hspace=0.01, bottom=0.1, top=0.92, left=0.05, right=0.98)
            cbar_ax = fig_abs.add_axes([0.05, 0.02, 0.93, 0.04])
            sm = plt.cm.ScalarMappable(cmap=cfg['abs_cmap'], norm=plt.Normalize(vmin=cfg['abs_vmin'], vmax=cfg['abs_vmax']))
            cbar = fig_abs.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend=extend_abs)
            cbar.set_label(f"{var_key} - {cfg['unit']}", fontsize=16, fontweight='bold')
            # --- TAMBAHKAN BARIS INI UNTUK MEMBESARKAN FONT TICK COLORBAR ---
            cbar.ax.tick_params(labelsize=14)
            fig_abs.savefig(os.path.join(self.output_dir, f"SPATIAL_ABSOLUTE_{var_key.upper()}.png"), dpi=300, bbox_inches='tight')
            plt.close(fig_abs)

            # 2. RENDER PLOT BIAS
            fig_bias, axes_bias = plt.subplots(4, 4, figsize=(16, 8), subplot_kw={'projection': ccrs.PlateCarree()})
            for i, season in enumerate(self.seasons):
                obs_season = obs.sel(season=season)
                for j in range(4):
                    ax = axes_bias[i, j]
                    self.prep_map(ax)
                    bias = models[j+1].sel(season=season) - obs_season
                    bias.plot.imshow(x='lon', y='lat', ax=ax, transform=self.data_crs, 
                                     vmin=cfg['bias_vmin'], vmax=cfg['bias_vmax'], 
                                     cmap='bwr', add_colorbar=False, 
                                     interpolation='bilinear', zorder=1, add_labels=False)
                    self.format_layout(ax, i, j, season, self.bias_titles[j])

            plt.subplots_adjust(wspace=0.01, hspace=0.01, bottom=0.1, top=0.92, left=0.05, right=0.98)
            cbar_ax = fig_bias.add_axes([0.05, 0.02, 0.93, 0.04])
            sm = plt.cm.ScalarMappable(cmap='bwr', norm=plt.Normalize(vmin=cfg['bias_vmin'], vmax=cfg['bias_vmax']))
            cbar = fig_bias.colorbar(sm, cax=cbar_ax, orientation='horizontal', extend='both')
            cbar.set_label(f"Bias {var_key} - {cfg['unit']}", fontsize=16, fontweight='bold')
            # --- TAMBAHKAN BARIS INI UNTUK MEMBESARKAN FONT TICK COLORBAR ---
            cbar.ax.tick_params(labelsize=14)
            fig_bias.savefig(os.path.join(self.output_dir, f"SPATIAL_BIAS_{var_key.upper()}.png"), dpi=300, bbox_inches='tight')
            plt.close(fig_bias)

    def run_pipeline(self):
        for var_key in self.config.keys():
            self.plot_variable(var_key)
        print("🎉 [SUCCESS] Absolute & Bias Matrices generated!")

if __name__ == "__main__":
    SpatialAbsoluteBiasPlotter().run_pipeline()

In [ ]:
"""
Project: Sovereign Climate Bias Correction - CDF Plotter V3.3
Description: Visualisasi Cumulative Distribution Function (CDF) dari hasil koreksi bias (QDM).
             Solusi Overlapping Curves (Lama vs Baru):
             - Stacked Linewidth: Line Lama lebih tebal (3.2) + transparan (alpha=0.55).
             - Line Baru ditaruh di atasnya dengan tebal sedang (1.6) + alpha=0.95.
             - Path input CSV disesuaikan ke direktori terbaru.
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

class LocalCDFPlotter:
    def __init__(self):
        # ⚡ PATH INPUT MASTER CSV SESUAI REQUEST LU ⚡
        self.csv_pdf = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\3-bias_corr\data\Titik_Koordinat_PDF_Master_ALL_AREA_FULL.csv"
        self.out_dir = r"C:\IPBUniversity\CoolYeah\FASTTRACK\TESIS\data\3_bias_correction\output_plots\rrtm_ncld1"
        os.makedirs(self.out_dir, exist_ok=True)

    def _draw_cdf_figure(self, df_vs, var, scen, ts, ts_label, plot_type_mode='standard'):
        """
        Helper function internal untuk merender 1 kanvas grafik CDF.
        plot_type_mode: 'standard', 'old_only', 'new_only', 'combined'
        """
        plt.figure(figsize=(10, 6), dpi=300)

        # 1. Label Legend Dinamis
        if plot_type_mode == 'combined':
            label_ec_old  = 'EC-Earth3 (Corrected Lama)'
            label_ec_new  = 'EC-Earth3 (Corrected Baru)'
            label_nor_old = 'NorESM (Corrected Lama)'
            label_nor_new = 'NorESM (Corrected Baru)'
        else:
            label_ec_old  = 'EC-Earth3 (Corrected)'
            label_ec_new  = 'EC-Earth3 (Corrected)'
            label_nor_old = 'NorESM (Corrected)'
            label_nor_new = 'NorESM (Corrected)'

        # 2. Plot Observasi (ERA5) - Hitam Solid (Zorder Paling Atas)
        obs = df_vs[df_vs['Type'] == 'Obs']
        if not obs.empty:
            plt.plot(obs['X'], obs['Y_CDF'], color='black', linestyle='-', linewidth=2.5, 
                     label='ERA5', zorder=10)

        # 3. Filter Data Model EC-Earth3 & NorESM
        ec_raw = df_vs[(df_vs['Model'] == 'EC-Earth3') & (df_vs['Type'] == 'Raw')]
        ec_old = df_vs[(df_vs['Model'] == 'EC-Earth3') & (df_vs['Type'].isin(['Corrected', 'Corrected_Old']))]
        ec_new = df_vs[(df_vs['Model'] == 'EC-Earth3') & (df_vs['Type'] == 'Corrected_New')]

        nor_raw = df_vs[(df_vs['Model'] == 'NorESM') & (df_vs['Type'] == 'Raw')]
        nor_old = df_vs[(df_vs['Model'] == 'NorESM') & (df_vs['Type'].isin(['Corrected', 'Corrected_Old']))]
        nor_new = df_vs[(df_vs['Model'] == 'NorESM') & (df_vs['Type'] == 'Corrected_New')]

        # ==============================================================================
        # DRAW EC-EARTH3 (Warna: Merah)
        # ==============================================================================
        # Raw -> Merah Solid Agak Transparan
        if not ec_raw.empty: 
            plt.plot(ec_raw['X'], ec_raw['Y_CDF'], color='red', linestyle='-', linewidth=1.2, 
                     alpha=0.6, label='EC-Earth3 (Raw)', zorder=2)
        
        # Corrected Lama -> Merah Putus-putus TEBAL (linewidth=3.2) & TRANSPARAN (alpha=0.55)
        if plot_type_mode in ['standard', 'old_only', 'combined']:
            if not ec_old.empty: 
                plt.plot(ec_old['X'], ec_old['Y_CDF'], color='red', linestyle='--', linewidth=3.2, 
                         alpha=0.55, label=label_ec_old, zorder=4)
        
        # Corrected Baru -> Merah Dash-dot TIPIS (linewidth=1.6) & TEGAS (alpha=0.95)
        if plot_type_mode in ['new_only', 'combined']:
            if not ec_new.empty: 
                ls_style = ':' if plot_type_mode == 'combined' else '--'
                lw_style = 1.6 if plot_type_mode == 'combined' else 2.0
                plt.plot(ec_new['X'], ec_new['Y_CDF'], color='darkred' if plot_type_mode == 'combined' else 'red', 
                         linestyle=ls_style, linewidth=lw_style, alpha=0.95, 
                         label=label_ec_new, zorder=6)

        # ==============================================================================
        # DRAW NORESM (Warna: Biru)
        # ==============================================================================
        # Raw -> Biru Solid Agak Transparan
        if not nor_raw.empty: 
            plt.plot(nor_raw['X'], nor_raw['Y_CDF'], color='blue', linestyle='-', linewidth=1.2, 
                     alpha=0.6, label='NorESM (Raw)', zorder=2)
        
        # Corrected Lama -> Biru Putus-putus TEBAL (linewidth=3.2) & TRANSPARAN (alpha=0.55)
        if plot_type_mode in ['standard', 'old_only', 'combined']:
            if not nor_old.empty: 
                plt.plot(nor_old['X'], nor_old['Y_CDF'], color='blue', linestyle='--', linewidth=3.2, 
                         alpha=0.55, label=label_nor_old, zorder=3)
        
        # Corrected Baru -> Biru Dash-dot TIPIS (linewidth=1.6) & TEGAS (alpha=0.95)
        if plot_type_mode in ['new_only', 'combined']:
            if not nor_new.empty: 
                ls_style = ':' if plot_type_mode == 'combined' else '--'
                lw_style = 1.6 if plot_type_mode == 'combined' else 2.0
                plt.plot(nor_new['X'], nor_new['Y_CDF'], color='navy' if plot_type_mode == 'combined' else 'blue', 
                         linestyle=ls_style, linewidth=lw_style, alpha=0.95, 
                         label=label_nor_new, zorder=5)

        # ==============================================================================
        # KUSTOMISASI GRAFIK
        # ==============================================================================
        suffix_title = ""
        if plot_type_mode == 'old_only': suffix_title = " [Metode Lama]"
        elif plot_type_mode == 'new_only': suffix_title = " [Metode Baru]"
        elif plot_type_mode == 'combined': suffix_title = " [Komparasi Metode]"

        plt.title(f"CDF Validation ({ts_label}): {var.upper()} - {scen.upper()}{suffix_title}", fontsize=13, fontweight='bold')
        unit = "Celcius" if var == 't2m' else "W/m2" if var == 'rsds' else "m/s"
        plt.xlabel(f"{var} ({unit})", fontsize=11, fontweight='bold')
        plt.ylabel("Cumulative Probability", fontsize=11, fontweight='bold')

        plt.ylim(0, 1.05)
        if var in ['rsds', 'ws10', 'ws100']: plt.xlim(left=0)

        plt.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
        plt.grid(True, linestyle=':', alpha=0.6)

        # Penamaan File Output
        file_tag = ""
        if plot_type_mode == 'old_only': file_tag = "_OLD"
        elif plot_type_mode == 'new_only': file_tag = "_NEW"
        elif plot_type_mode == 'combined': file_tag = "_COMBINED"

        out_file = os.path.join(self.out_dir, f"SmoothCDF_{ts}_{var}_{scen}{file_tag}.png")
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved: {os.path.basename(out_file)}")

    def plot_all(self, target_time_scales=None):
        print("🚀 Membaca Master Data CSV untuk Plotting CDF...")
        if not os.path.exists(self.csv_pdf):
            print(f"❌ ERROR: File CSV tidak ditemukan di {self.csv_pdf}")
            return
            
        df = pd.read_csv(self.csv_pdf)

        if 'Y_CDF' not in df.columns:
            print("❌ ERROR: Kolom 'Y_CDF' tidak ditemukan! Harap jalankan ulang extract_pdf_metrics.py.")
            return

        # ⚡ PENYELAMAT FISIKA SUHU (Kelvin ke Celsius)
        kelvin_mask = (df['Variable'] == 't2m') & (df['X'] > 200)
        df.loc[kelvin_mask, 'X'] -= 273.15
        
        double_minus_mask = (df['Variable'] == 't2m') & (df['X'] < -100)
        df.loc[double_minus_mask, 'X'] += 273.15

        if 'TimeScale' in df.columns:
            available_time_scales = df['TimeScale'].dropna().unique().tolist()
        else:
            df['TimeScale'] = 'Daily'
            available_time_scales = ['Daily']

        if target_time_scales is not None:
            if isinstance(target_time_scales, str):
                target_time_scales = [target_time_scales]
            time_scales_to_process = [ts for ts in target_time_scales if ts in available_time_scales]
        else:
            time_scales_to_process = available_time_scales

        variables = df['Variable'].unique()
        scenarios = df['Scenario'].unique()

        for ts in time_scales_to_process:
            df_ts = df[df['TimeScale'] == ts]
            ts_label = "6-Hourly" if ts == '6hr' else "Daily"
            
            print(f"\n==========================================================")
            print(f"📊 MEMPROSES PLOT CDF UNTUK SKALA WAKTU: {ts_label} ({ts})")
            print(f"==========================================================")

            for var in variables:
                for scen in scenarios:
                    df_vs = df_ts[(df_ts['Variable'] == var) & (df_ts['Scenario'] == scen)]
                    if df_vs.empty: continue

                    # ⚡ PERCABANGAN KHUSUS RSDS SSP585 ⚡
                    if var.lower() == 'rsds' and scen.lower() == 'ssp585':
                        print(f"📈 [SPECIAL 3-WAY PLOT] RSDS - SSP585 ({ts_label}):")
                        self._draw_cdf_figure(df_vs, var, scen, ts, ts_label, plot_type_mode='old_only')
                        self._draw_cdf_figure(df_vs, var, scen, ts, ts_label, plot_type_mode='new_only')
                        self._draw_cdf_figure(df_vs, var, scen, ts, ts_label, plot_type_mode='combined')
                    else:
                        print(f"📈 Menggambar CDF [{ts}]: {var.upper()} - {scen.upper()}")
                        self._draw_cdf_figure(df_vs, var, scen, ts, ts_label, plot_type_mode='standard')

        print("\n🎉 [SUCCESS] Seluruh Plot CDF berhasil dirender!")

if __name__ == "__main__":
    plotter = LocalCDFPlotter()
    plotter.plot_all()